# SteamCharts -> Silver y DLQ

Convierte `Mon-YY` a columnas temporales y conserva las filas válidas en Silver. Los meses o métricas imposibles se guardan en DLQ.

In [ ]:
import json
dbutils.widgets.text("catalog", "steam_game_intelligence")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("dlq_schema", "dlq")
dbutils.widgets.text("cutoff_us", "")
dbutils.widgets.text("run_id", "")
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
dlq_schema = dbutils.widgets.get("dlq_schema")
cutoff_us = dbutils.widgets.get("cutoff_us")
run_id = dbutils.widgets.get("run_id")

In [ ]:
%sql
-- Silver: una medición válida por app_id y mes.
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :silver_schema || '.player_counts_monthly') AS
WITH parsed AS (
  SELECT * EXCEPT (month), trim(try_cast(month AS STRING)) AS month_raw, try_cast(steam_appid AS BIGINT) AS app_id,
    try_cast(avg_players AS DECIMAL(18,2)) AS avg_players_typed,
    try_cast(peak_players AS BIGINT) AS peak_players_typed,
    try_cast(try_cast(gain AS STRING) AS DECIMAL(18,4)) AS gain_typed,
    try_cast(gain_percent AS DECIMAL(18,8)) AS gain_percent_typed,
    lower(substr(trim(try_cast(month AS STRING)), 1, 3)) AS month_name,
    try_cast(right(trim(try_cast(month AS STRING)), 2) AS INT) AS year_short
  FROM IDENTIFIER(:catalog || '.' || :bronze_schema || '.steamcharts_2025_history_data')
  WHERE _ingested_at <= timestamp_micros(try_cast(:cutoff_us AS BIGINT))
), typed AS (
  SELECT *,
    CASE month_name WHEN 'jan' THEN 1 WHEN 'feb' THEN 2 WHEN 'mar' THEN 3 WHEN 'apr' THEN 4 WHEN 'may' THEN 5 WHEN 'jun' THEN 6 WHEN 'jul' THEN 7 WHEN 'aug' THEN 8 WHEN 'sep' THEN 9 WHEN 'oct' THEN 10 WHEN 'nov' THEN 11 WHEN 'dec' THEN 12 END AS month,
    CASE WHEN year_short <= 49 THEN 2000 + year_short ELSE 1900 + year_short END AS year
  FROM parsed
), dated AS (
  SELECT *, make_date(year, month, 1) AS month_start
  FROM typed
), catalog_match AS (
  SELECT d.*, g.app_id AS catalog_app_id
  FROM dated d LEFT JOIN IDENTIFIER(:catalog || '.' || :silver_schema || '.games') g ON d.app_id = g.app_id
), ranked AS (
  SELECT *, row_number() OVER (PARTITION BY app_id, month_start ORDER BY _ingested_at DESC, sha2(to_json(struct(*)), 256) DESC) AS rn
  FROM catalog_match
), checked AS (
  SELECT *, filter(array(
    CASE WHEN app_id IS NULL OR app_id <= 0 THEN 'app_id_invalido' END,
    CASE WHEN month_start IS NULL OR year < 1950 OR year > 2049 THEN 'mes_no_parseable' END,
    CASE WHEN avg_players_typed < 0 OR peak_players_typed < 0 THEN 'jugadores_negativos' END,
    CASE WHEN avg_players_typed > peak_players_typed THEN 'avg_mayor_que_peak' END,
    CASE WHEN trim(try_cast(gain AS STRING)) <> '' AND gain_typed IS NULL THEN 'gain_no_parseable' END
  ), x -> x IS NOT NULL) AS error_reasons
  FROM ranked WHERE rn = 1
)
SELECT app_id, month_raw, year, month, 1 AS day, month_start,
  avg_players_typed AS avg_players, peak_players_typed AS peak_players,
  gain_typed AS gain, gain_percent_typed AS gain_percent, name,
  CASE WHEN catalog_app_id IS NOT NULL THEN true ELSE false END AS in_catalog,
  _ingested_at, _source
FROM checked WHERE size(error_reasons) = 0;

In [ ]:
%sql
-- DLQ: conserva los meses y métricas que no pudieron validarse.
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :dlq_schema || '.player_counts_monthly_rejected') AS
WITH parsed AS (
  SELECT * EXCEPT (month), trim(try_cast(month AS STRING)) AS month_raw, try_cast(steam_appid AS BIGINT) AS app_id_typed,
    try_cast(avg_players AS DECIMAL(18,2)) AS avg_players_typed,
    try_cast(peak_players AS BIGINT) AS peak_players_typed,
    try_cast(try_cast(gain AS STRING) AS DECIMAL(18,4)) AS gain_typed,
    lower(substr(trim(try_cast(month AS STRING)), 1, 3)) AS month_name,
    try_cast(right(trim(try_cast(month AS STRING)), 2) AS INT) AS year_short
  FROM IDENTIFIER(:catalog || '.' || :bronze_schema || '.steamcharts_2025_history_data')
  WHERE _ingested_at <= timestamp_micros(try_cast(:cutoff_us AS BIGINT))
), dated AS (
  SELECT *, CASE month_name WHEN 'jan' THEN 1 WHEN 'feb' THEN 2 WHEN 'mar' THEN 3 WHEN 'apr' THEN 4 WHEN 'may' THEN 5 WHEN 'jun' THEN 6 WHEN 'jul' THEN 7 WHEN 'aug' THEN 8 WHEN 'sep' THEN 9 WHEN 'oct' THEN 10 WHEN 'nov' THEN 11 WHEN 'dec' THEN 12 END AS month,
    CASE WHEN year_short <= 49 THEN 2000 + year_short ELSE 1900 + year_short END AS year
  FROM parsed
), checked AS (
  SELECT *, make_date(year, month, 1) AS month_start,
    filter(array(
      CASE WHEN app_id_typed IS NULL OR app_id_typed <= 0 THEN 'app_id_invalido' END,
      CASE WHEN make_date(year, month, 1) IS NULL OR year < 1950 OR year > 2049 THEN 'mes_no_parseable' END,
      CASE WHEN avg_players_typed < 0 OR peak_players_typed < 0 THEN 'jugadores_negativos' END,
      CASE WHEN avg_players_typed > peak_players_typed THEN 'avg_mayor_que_peak' END,
      CASE WHEN trim(try_cast(gain AS STRING)) <> '' AND gain_typed IS NULL THEN 'gain_no_parseable' END
    ), x -> x IS NOT NULL) AS error_reasons
  FROM dated
)
SELECT sha2(concat_ws('|', coalesce(cast(app_id_typed AS STRING), 'NULL'), coalesce(cast(month_start AS STRING), 'NULL'), to_json(struct(*))), 256) AS _reject_key,
  app_id_typed AS app_id, month, year, month_start,
  array_join(error_reasons, '; ') AS motivo_rechazo, error_reasons AS reglas_violadas,
  to_json(struct(*)) AS payload, :run_id AS _run_id, current_timestamp() AS _rejected_at
FROM checked WHERE size(error_reasons) > 0;

In [ ]:
dbutils.notebook.exit('{"status":"completed","table":"player_counts_monthly"}')